# Chicken Detection Negative Fine-tune

Notebook nay fine-tune model detect ga hien tai voi bo `merged_dataset` moi da them anh am.

Nguon dau vao mac dinh tren Google Drive:
- `MyDrive/chicken_detection/merged_dataset.zip`
- `MyDrive/chicken_detection/runs/detect/chicken_detection/weights/best.pt`

Ket qua se duoc luu tai:
- `MyDrive/chicken_detection/runs/detect/chicken_detection_neg_v1/weights/best.pt`


In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path
import shutil
import zipfile

In [ ]:
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/chicken_detection')
DATASET_ZIP = DRIVE_ROOT / 'merged_dataset.zip'
BEST_MODEL = DRIVE_ROOT / 'runs' / 'detect' / 'chicken_detection' / 'weights' / 'best.pt'
DATASET_ROOT = Path('/content/merged_dataset')
RUNS_ROOT = DRIVE_ROOT / 'runs' / 'detect'
RUN_NAME = 'chicken_detection_neg_v1'

EPOCHS = 20
IMG_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 8
LEARNING_RATE = 0.003

print('Dataset zip :', DATASET_ZIP)
print('Base model  :', BEST_MODEL)
print('Output run  :', RUNS_ROOT / RUN_NAME)

if not DATASET_ZIP.exists():
    raise FileNotFoundError(f'Khong tim thay dataset zip: {DATASET_ZIP}')

if not BEST_MODEL.exists():
    raise FileNotFoundError(f'Khong tim thay best.pt de fine-tune: {BEST_MODEL}')

In [ ]:
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
    zip_ref.extractall('/content')

data_yaml = DATASET_ROOT / 'data.yaml'
if not data_yaml.exists():
    raise FileNotFoundError(f'Khong tim thay data.yaml sau khi giai nen: {data_yaml}')

print('Dataset san sang tai:', DATASET_ROOT)
print('Data yaml:', data_yaml)

In [ ]:
model = YOLO(str(BEST_MODEL))

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    patience=PATIENCE,
    lr0=LEARNING_RATE,
    save=True,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

print('Training xong.')

In [ ]:
best_finetuned = RUNS_ROOT / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(str(best_finetuned))

val_metrics = best_model.val(data=str(data_yaml))
test_metrics = best_model.val(data=str(data_yaml), split='test')

print('=' * 80)
print('KET QUA FINE-TUNE')
print('=' * 80)
print('Best model moi:', best_finetuned)
print(f'Validation mAP50    : {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95 : {val_metrics.box.map:.4f}')
print(f'Validation Precision: {val_metrics.box.mp:.4f}')
print(f'Validation Recall   : {val_metrics.box.mr:.4f}')
print(f'Test mAP50          : {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95       : {test_metrics.box.map:.4f}')
print(f'Test Precision      : {test_metrics.box.mp:.4f}')
print(f'Test Recall         : {test_metrics.box.mr:.4f}')